In [1]:
import os
import sys

import numpy as np
import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
from legnet_embedding import LegNetEmbedding
sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [ ]:
utr_type = "utr3"
MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"

In [3]:
cell_types = ["c1", "c13", "c17", "c2", "c4", "c6"]

In [4]:
use_top_n = 1024
file_mask = '/home/kekulen/ml_projects/rna-legnet_big/rna-legnet-benchmark/figures/diffusion/trajectories/trajectories_{}.parquet'
files = {ct: file_mask.format(ct) for ct in cell_types}
df = pd.concat([pd.read_parquet(file).iloc[:use_top_n].assign(gen_cell_type=ct) for ct, file in files.items() if os.path.isfile(file)])
df

,0,1,2,3,4,5,6,7,8,9,...,140,141,142,143,144,145,146,147,148,gen_cell_type
0,CGTTGGAGGGCCTATGAGCTGAGCTAGGCATGGTGTGAGCCGTCGG...,ATCACGTCGGCCTGGGAGCGGATCTAGGCATAGTGGGTCGCGCGGC...,GGGTAGTTGGGGGGGGAGAAGCGGATGGTAATGTCTGTCAAAGAAC...,CGGGTGGAGTGGGGCGGGTGGAAGAAGGATAACGCAAACAGAGAAG...,AGGGGGGAGGGGAGAGGTATGAGTATGGCTAAGAGAAGATGATAGA...,CGGGGGGGGGATACAATGGGAGGAGGGGCGGGGAGAAGGCGTGATA...,AAGGTGGGGCAGCATAGATATGACGGTGTCTAGTGAACGATGTAGA...,CGGCGAAGGAAGGAAAAAGAGGGGCCCGAAATGAGTAGAAAGGGCA...,GAAAGGGGTAAGTAGGAAGGATAGTAGGGAATGAGAGGGCGGGGGC...,ACCACCGAAAAGGAGTAAGGACTTAGGTGTGGTGGGGTGGAGCGGA...,...,CCTAAGTGCCAGGGAATTACAAGAACGAGGGGGAGGCGAGGGCGTG...,CCTAAGTGCCAGGGAATTACAAGAAGGAGGGGGAGGCGAGGGCGTG...,CCTAAGTGCCAGGGAATGACAAGAAGGAGGGGGAGGCGAGGGCGTG...,CCTAAGTGCCAGGGAATGACAAGAAGAAGGGGGAGGGGAGGGCGGG...,CCTGAGTGCCAGGGAATGACAAGAAGAAGGGGGAGGGGAGGGCGGG...,CCTGAGTGCCAGGGAATGACGAGAAGAAGGGGGAGGGGAGGGCGGG...,CCTGAGTGCCAGGGAATGACGAGAAGAAGGGGGAGGGGAGGGCGGG...,CCTGAGTGCCAGGGAATGACGAGAAGAAGGGGGAGGGGAGGGCGGG...,CCTGAGTGCCAGGGAATGACGAGAAGAAGGGGGAGGGGAGGGCGGG...,c13
1,GGAATTATTTTACAACCCGACTTATATTGGAGTCATCCCCGGGTGA...,AGAAATATGTTACAAACCGTCTCATCTTGAAGCGATTACGAGCTGT...,AGTACTATATTAACAACTATCTTTTCTCAAACCAATAAAACTATAT...,ATCATAATATTGAGATAAGTAGTGTTGCAATTAAATATAAAACAAT...,TCCAAAGGACCCTAATAACTTTTATTTCAATTACGCACAAAATAAA...,TCTACAATAAATTAATAAATTTCATTCAAATTTACAATCAAATAAA...,TATGATATAAAAGAAAAGAGTTGGTAAGGATGAGAATCAAGAAACA...,TATCAGAAAAAAAAGAATAGATAGCGTTAGAGAGATATAATAAGGA...,AATGTCACACAGAAAAGAAAGAGAATTACAATACATATACAGTTGA...,AAAACGAACGAAAAAATGTTTCTCATTAGAATAAACATACATTTAA...,...,AGGTTGAGCCCAGGGGGAACACTGTTACCCTCTGAGAGGAAAACTA...,AGGTTGAGCCCAGGGGGAACAATGTTACCCTCGGAGAGGAAAACTA...,AGGTTGAGCCCAGGGGGAACAATGTTACCCTCGGAGAGGAAAACTA...,AGGTTGAGCCCAGGGGGAACAATGTTACCCTCGGAGAGGAAAACTA...,AGGTTGAGCCTAGGGGGAACAATGTTACCCTGGGAGAGGAAAACTA...,AGGTTGAGCCTAGGGGGAACAATGTTACCCTGGGAGAGGAAAACTA...,AGGTTGAGCCTAGGGGGAACAATGTTACCCTGGGAGAGGAAAACTA...,AGGTTGAGCCTAGGGGGAACAATGTTACCCTAGGAGAGGAAAACTA...,AGGTTGAGCCTAGGGGGAACAATGTTACCCTAGGAGAGGAAAACTA...,c13
2,ATCACAACGGCATATTGAAAGTTTACAGACAAGAGTCTAGTGACGG...,TGAATTATTGCATAGTAGCCATCTATAAACGATAATCACGGCGATT...,GAAATTATCTGATATGTGAAATATACCAATAAATGTAAAGACGATC...,AAAATAGACTAATTATCACAATATTATAATAAAAATAAAAAACATA...,TAAGAATATTAATGGTACGCAACTTATGAAATACTTCAAAGATATT...,AAGTAATGTTAGTTTTACATAAGTTGAAAAACATGTAGACCATATT...,CCATCAAAAGATTATTAGAATAATTACAAAATACATAAACGATATA...,GAACCATAAAGCGATCTTATTTATTTTAAAATAACAAAAATAGACT...,TATAAATAACAGATATTTTTTTAATTTAAAAAAGACTAAAGAAGTA...,GCTAAATGAGATATATGTTAGAATTTTAAAAGGTAAAAAAAAAAAA...,...,CACACACACACACACATATATATATATATGTGTGTGTGTCTGTGTG...,CACACACACACACACGTCTATATATATATGTGTGTGTGTGTGTATG...,CACACACACACACACGTATATATATATATGTGTGTGTGTGTGTATG...,CACACACACACACACCTATATATATATATGTGTGTGTGTGTGTGTG...,CACACACACACACACATATATATATATATGTGTGTGTGTGTGTGTG...,CACACACACACACACATATATATATATATGTGTGTGTGTGTGTGTG...,CACACACACACGCACATATATATATATATGTGTGTGTGTGTGTGTG...,CACACACACACACACATATATATCTATATGTGTGTGTGTGTGTGTG...,CACACACACACACACATATATATATATATGTGTGTGTGTGTGTGTG...,c13
3,TTTGGGGCATAAGGAATAGATTAAGCACAGTGACCATTCGGTACAG...,GTGTGGGCATGAAGACTACAGTAAGCGCACCGCCTTGGCCCAACTG...,GTCGCTCCAGAGACAAAACAGGTAGCGAACATCTATGGGCCACATG...,AGCACCCAACAGTCGATGTGGAAAGAGAACCCCCACGCTGTACACT...,ACGGACCTAGAGGGGATCAGGCACAAGGACCACAAAGTAGAACAGG...,TACGCCCAAGAGGTGAAATGACACACCTACAAGAAAACCGATCAGC...,AGGTCTAAGCAGAAGAAGAGACAGAGCGGCATCCATCACCTGGAGG...,ATGTTCAAGCAGAAGCCGACCCACACCTGCACCCACCGCCCCGACG...,TTGGAGACGACGGATCAGATCTGACAGAGAAGCGGCAACTCAGACA...,AGAGGGAAAACATGGGAGACCTAAGTAGGTGAAAGCCACGGGATAA...,...,GCACACCCAGGCACTCGATGACTGGACGGAGAGAGGTGAGGTGCTG...,GTACACCCAGGGACTCGATGACGGGACGGAGAGAGGTGAGGTGCTG...,GTACATCCAGGGACTCGATGACGGGAAGGAGAGAGGTGAGGTGCTG...,GTACATCCAGGGACTCGATGACGGGAAGGAGAGAGGTGAGGTGCTG...,GTACATCCAGGGACTCGATGCCGGGAAGGAGAGAGGTGAGGTGCTG...,GTACATCCAGGGACTCGATGCCGGGAAGAAGAGAGGTGAGGTGCTG...,GTACATCCAGGGACTCGATGCCGGGAAGAAGAGAGGTGAGGTGCTG...,GTACATCCAGGGACTCGATGCCGGGAAGAAGAGAGGTGAGGTGCTG...,GTACATCCAGGGACTCGATGCCGGGAAGAAGAGAGGTGAGGTGCTG...,c13
4,CCCACGCTAGTCGCAGTAAAGGGCAGACCAATAAGCGGATGCCCAG...,ACACCGCTAGGCGCGGTTACGGGGAGAGCACGA

In [5]:
df['gen_cell_type'].unique()

array(['c13', 'c17', 'c2', 'c4', 'c6'], dtype=object)

In [6]:
sequences = df.rename_axis('seq_no').reset_index().melt(id_vars=['seq_no', 'gen_cell_type'], var_name='step', value_name='seq')
sequences

,seq_no,gen_cell_type,step,seq
0,0,c13,0,CGTTGGAGGGCCTATGAGCTGAGCTAGGCATGGTGTGAGCCGTCGG...
1,1,c13,0,GGAATTATTTTACAACCCGACTTATATTGGAGTCATCCCCGGGTGA...
2,2,c13,0,ATCACAACGGCATATTGAAAGTTTACAGACAAGAGTCTAGTGACGG...
3,3,c13,0,TTTGGGGCATAAGGAATAGATTAAGCACAGTGACCATTCGGTACAG...
4,4,c13,0,CCCACGCTAGTCGCAGTAAAGGGCAGACCAATAAGCGGATGCCCAG...
...,...,...,...,...
762875,1019,c6,148,TGGTAAAGCCAGCATGGGGTATCTGATCCTGACTCACTTTTGATAT...
762876,1020,c6,148,AGACAGATGTTCAAGGTGTAGGAATGGTTATAAGCAGAGGTGGTAT...
762877,1021,c6,148,CTCTGTCAGCGCATGTGTACGTGCGTGCGCGTGTACCTGCATGCCC...
762878,1022,c6,148,CCTGCCCACACACACACACACACACACACACACACACATACACACA...


In [7]:
df_ct = pd.concat([sequences.copy().assign(cell_type=ct) for ct in cell_types]).sort_values(by='seq')
df_ct

,seq_no,gen_cell_type,step,seq,cell_type
758082,322,c13,148,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,c13
758082,322,c13,148,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,c17
758082,322,c13,148,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,c4
758082,322,c13,148,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,c2
758082,322,c13,148,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...,c1
...,...,...,...,...,...
717671,871,c13,140,TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTATT...,c2
717671,871,c13,140,TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTATT...,c1
717671,871,c13,140,TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTATT...,c17
717671,871,c13,140,TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTATT...,c13


In [9]:
ct_classes = df_ct["cell_type"].unique()
num_classes = ct_classes.shape[0]
num_classes

6

In [10]:
batch_size = 1024

In [11]:
num_workers = 32

In [12]:
test_set = utrdata.UTRData(
    df=df_ct,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)

In [13]:
# Creating DataLoaders
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

In [14]:
ckpt = torch.load(MODEL_PATH)
ckpt["hyper_parameters"]["model_class"] = LegNetEmbedding
loaded_model = RNARegressor(**ckpt["hyper_parameters"])
loaded_model.load_state_dict(ckpt["state_dict"])

In [17]:
progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)
trainer = pl.Trainer(
    callbacks=[progressbar_callback],
    logger=False,
    accelerator="gpu",
    devices=[1],
    deterministic=True,
)

prediction = trainer.predict(model=loaded_model, dataloaders=dl_test)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [18]:
pred_tuples, _ = zip(*prediction)
embed, pred = zip(*pred_tuples)
embed = torch.concat(embed).numpy()
pred = torch.concat(pred).numpy()

In [19]:
seq_embedding = embed.reshape(-1, num_classes * embed.shape[-1], order="C")

In [20]:
pred_mass_center = pred[:, 1].reshape(-1, num_classes)

In [21]:
result = pd.DataFrame(
    pred_mass_center,
    columns=pd.MultiIndex.from_product([["pred_mass_center"], cell_types]),
    index=df_ct.iloc[::num_classes]["seq"]
)
result.shape

(762880, 6)

In [22]:
metainf_df = sequences.set_index('seq').sort_index()
metainf_df.columns = pd.MultiIndex.from_product([['traject_data'], metainf_df.columns])
result = pd.concat([result, metainf_df], axis=1)
result.shape

(762880, 9)

In [23]:
seq_embedding_df = pd.DataFrame(
    seq_embedding,
    columns=pd.MultiIndex.from_product([
        ["embedding"],
        [f"{ct}_{i:03d}" for ct in result["pred_mass_center"].columns for i in range(embed.shape[-1])]
    ]),
    index=result.index)
result = pd.concat([result, seq_embedding_df], axis=1)
result.shape

(762880, 201)

In [24]:
result["embedding"]

,c1_000,c1_001,c1_002,c1_003,c1_004,c1_005,c1_006,c1_007,c1_008,c1_009,...,c6_022,c6_023,c6_024,c6_025,c6_026,c6_027,c6_028,c6_029,c6_030,c6_031
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAGGCACACACACACATGCACACACACACACACACACACAGGAGAGAGAAAATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCGCACACACACACACACACAGCTTTTGAAAACGCAGTTCTGAGACAC,0.060973,0.074850,0.144576,-0.001261,0.175742,0.191176,0.020255,0.088053,0.156706,0.076705,...,0.211980,0.171066,0.209862,0.151507,0.056383,0.061170,-0.054934,0.109796,-0.011600,0.084246
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAAACTATTTTTTTTTTAAAAACAAAAACTAGATACATAAGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGGGTGTGTGTGTGTGTGTGTGTGTGCACACACTCCAGCCACTAATGGACCCTATGAATATAAGAAAAACAAAACAACAAACCAAA,-0.011496,0.078013,0.138144,0.019924,0.166993,0.181923,0.033784,0.104986,0.131469,0.057433,...,0.173482,0.161416,0.232084,0.125498,0.036458,0.043460,-0.032546,0.086666,-0.037606,0.064834
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAAACTATTTTTTTTTTAAAAACAAAAACTAGATACATAAGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCACACACTACAGGCACTAATGGACACTATGAATATAAGAAAAACAAAACAACAAACCAAA,0.119792,0.044025,0.210335,0.090158,0.182483,0.147753,0.058386,0.083461,0.101896,0.028540,...,0.198290,0.196292,0.186381,0.086310,-0.003370,-0.019966,0.054856,-0.017346,-0.020356,0.011565
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAAACTATTTTTTTTTTAAAAACAAAAACTATATACATATGTGTGTGTGTGTGTGTGTGTGAGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCACACACTCCAGCCACTAATGGACCCTATGAATATAAAAAAAACAAAACAACAAACCAAA,-0.024116,0.068629,0.138482,0.015967,0.157048,0.191793,0.017067,0.090112,0.130688,0.050273,...,0.168128,0.162068,0.217679,0.109372,0.020568,0.028265,-0.047098,0.088258,-0.059710,0.047945
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAAACTATTTTTTTTTTAAAAACAAAAACTATATACATATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGAGTGTGTGTGTGTGTGTGTGTGTATGTGTGTGTGTGTGCACACACTCCAGCCACTAATGGACCCTATGAATATAAGAAAAACAAAACAACAAACCAAA,0.111182,0.040480,0.209623,0.093593,0.183906,0.160239,0.026795,0.082628,0.118822,0.033839,...,0.215265,0.127698,0.303201,0.141229,0.053721,0.079607,-0.013953,0.190557,-0.004886,0.053338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTAAAAAAAAAAAAAATAAAAAAAAAAGGTCGCTGATCCTGTGAGGGGCGGGAGGCTGGGAGTGGAAGGGCCAGGGTGAGGCGCCACGCCCGGGACCCAGCCAAGCCTTTCCCGCTCCACAGCCAGTTCCCTCACCACGTGAATCCACGAGACGTGCTGGGGTGTGACGGACGCCAGAGCTGGAGTTGCGGCAGCCGGGGGCTC,0.092331,0.112042,0.202104,0.207999,0.180390,0.072043,-0.039342,0.239288,0.069149,0.129576,...,0.157868,0.206862,0.235782,0.139970,0.094182,0.036097,0.164412,0.039509,0.074998,0.142474
TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTGTTTGTATGTACAAACATGTGTGTGTGTATATGTTTGTGTGTGTGTGTGTGCGTGTGTGGGTGTGTTTGGCTGTGTGTGTGTGCGTGTGTTTGTGTGTGTGTGTGTGTGCACACAAACACACACACACACACACACACACACACACACACACACACACACACAGACACACACACACACACACACCCGTGTGTGTGTCGG,0.137333,0.124442,0.182963,0.153783,0.109799,-0.024599,0.129865,0.373109,0.054756,0.010251,...,0.138183,0.126153,0.166271,0.109497,0.073595,0.028360,0.075060,-0.031501,-0.026711,0.123236
TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTGTTTGTATGTACACACATGTGGGTGTGTGTGTGTGCGTGTGTGTGCGTGTGTGTGTGTGTGTGTGTGTGTATGTGGGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCACACACACGCACACACACACACACACACACACACACACACACACACACACACACACACACACACACACACACACACGTATGTGTCTGTG,-0.002246,0.063185,0.177623,0.140346,0.132562,0.207101,-0.018514,0.164606,0.069646,0.107442,...,0.154876,0.113694,0.288130,0.124063,0.072322,0.110368,0.023105,0.194765,-0.039860,0.150860


### Counting k-mers

In [25]:
sys.path.append("../../benchmark/kmers")
from kmer_model import KmerLinearRegressor

In [26]:
kmer_dfs = []
for k in [1, 2, 3]:
    kmerreg = KmerLinearRegressor(kmer_length=k)
    kmer_df_k = kmerreg.count_kmers(result.index.values)
    kmer_df_k.columns = pd.MultiIndex.from_product([
        ["kmer_counts"],
        kmer_df_k.columns
    ])
    kmer_dfs.append(kmer_df_k)
    print(f"k={k} done")
result = pd.concat([result] + kmer_dfs, axis=1)

k=1 done
k=2 done
k=3 done


In [27]:
result["kmer_counts"]

,A,C,G,T,AA,AC,AG,AT,CA,CC,...,TCG,TCT,TGA,TGC,TGG,TGT,TTA,TTC,TTG,TTT
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAGGCACACACACACATGCACACACACACACACACACACAGGAGAGAGAAAATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCGCACACACACACACACACAGCTTTTGAAAACGCAGTTCTGAGACAC,99,34,57,50,62,27,8,2,29,0,...,0,1,2,2,0,41,0,1,1,2
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAAACTATTTTTTTTTTAAAAACAAAAACTAGATACATAAGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGGGTGTGTGTGTGTGTGTGTGTGTGCACACACTCCAGCCACTAATGGACCCTATGAATATAAGAAAAACAAAACAACAAACCAAA,106,22,50,62,80,14,4,7,12,5,...,0,0,1,1,2,39,1,0,0,8
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAAACTATTTTTTTTTTAAAAACAAAAACTAGATACATAAGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCACACACTACAGGCACTAATGGACACTATGAATATAAGAAAAACAAAACAACAAACCAAA,108,19,50,63,80,16,4,7,13,1,...,0,0,1,1,1,41,1,0,0,8
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAAACTATTTTTTTTTTAAAAACAAAAACTATATACATATGTGTGTGTGTGTGTGTGTGTGAGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCACACACTCCAGCCACTAATGGACCCTATGAATATAAAAAAAACAAAACAACAAACCAAA,107,22,47,64,81,14,2,9,12,5,...,0,0,2,1,1,40,1,0,0,8
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAAACTATTTTTTTTTTAAAAACAAAAACTATATACATATGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGAGTGTGTGTGTGTGTGTGTGTGTATGTGTGTGTGTGTGCACACACTCCAGCCACTAATGGACCCTATGAATATAAGAAAAACAAAACAACAAACCAAA,107,22,47,64,79,14,3,10,12,5,...,0,0,2,1,1,39,1,0,0,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTAAAAAAAAAAAAAATAAAAAAAAAAGGTCGCTGATCCTGTGAGGGGCGGGAGGCTGGGAGTGGAAGGGCCAGGGTGAGGCGCCACGCCCGGGACCCAGCCAAGCCTTTCCCGCTCCACAGCCAGTTCCCTCACCACGTGAATCCACGAGACGTGCTGGGGTGTGACGGACGCCAGAGCTGGAGTTGCGGCAGCCGGGGGCTC,53,54,69,64,25,9,16,3,12,19,...,1,0,5,2,4,2,1,2,1,37
TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTGTTTGTATGTACAAACATGTGTGTGTGTATATGTTTGTGTGTGTGTGTGTGCGTGTGTGGGTGTGTTTGGCTGTGTGTGTGTGCGTGTGTTTGTGTGTGTGTGTGTGTGCACACAAACACACACACACACACACACACACACACACACACACACACACACACAGACACACACACACACACACACCCGTGTGTGTGTCGG,46,44,53,97,4,37,1,4,37,2,...,1,0,0,3,2,39,0,0,5,43
TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTGTTTGTATGTACACACATGTGGGTGTGTGTGTGTGCGTGTGTGTGCGTGTGTGTGTGTGTGTGTGTGTGTATGTGGGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGCACACACACGCACACACACACACACACACACACACACACACACACACACACACACACACACACACACACACACACACGTATGTGTCTGTG,44,45,56,95,0,40,0,4,40,0,...,0,1,0,3,2,42,0,0,2,40


In [28]:
result.to_parquet(f"embeddings_mapperless_{utr_type}_trajectories_diffusion.parquet")

---